# Stress summary EDA: validity, recorded durations and activity
Prepared for Friday October 23, 2026. We read only `stress` and `activity` summary tables, not the millions of stress samples. The database and existing notebooks are unchanged. Outputs contain aggregate results and anonymous scatter points, not participant IDs or raw records.

**Questions:** Which stress values are available and usable? What relationships are observed among duration fields? Does recorded average stress co-vary with recorded steps, both pooled and within participants?

No source data dictionary is supplied with this project. Score scale, negative/sentinel meanings, qualifier rules and duration-category definitions remain **unverified**. Field names suggest seconds for duration and a dimensionless score, but names/SQL types alone cannot prove semantics. No clinical interpretation, diagnosis, causation or model validation follows.

## 1. Schema and row meaning
A candidate row is a participant's stress summary for one labelled reporting period. A synthetic example is a date label, reporting window, average score and several duration fields; it is not necessarily a complete day of valid stress measurement. Shared IDs/group/type provide context, date/week/day label the record, and start/end/duration describe the window.

Inspect every field before calculating. Missingness is not the only invalidity: negative scores and zero durations need separate attention. Credentials stay in `.env`; real rows are never displayed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately.'
engine=create_engine(url); tables={}; schema=[]
try:
    inspector=inspect(engine)
    for name in ['stress','activity']:
        for c in inspector.get_columns(name):
            schema.append({'table':name,'column':c['name'],'sql_type':str(c['type'])})
        with engine.connect() as conn:
            tables[name]=pd.read_sql(text('SELECT * FROM '+name),conn)
finally:
    engine.dispose()
del url
for df in tables.values(): df['calendar_date']=pd.to_datetime(df.calendar_date,errors='coerce')
s=tables['stress'].copy()
scores=['average_stress_level','max_stress_level']
durations=['stress_duration_sec','rest_stress_duration_sec','low_stress_duration_sec',
 'medium_stress_duration_sec','high_stress_duration_sec','activity_stress_duration_sec']
display(pd.DataFrame(schema).query("table == 'stress'"))
display(pd.DataFrame({'column':s.columns,'pandas_type':s.dtypes.astype(str).values,
 'missing_rows':s.isna().sum().values,'distinct_nonnull':s.nunique().values}))
print('Stress records:',len(s),'Participants:',s.sdv_user_id.nunique(),'Missing cells:',int(s.isna().sum().sum()))


## 2. Candidate keys, scores, durations and qualifier categories
Test record IDs, participant/date and participant/period keys. A key must be unique and non-null on both sides of any join. Do not infer matching from equal lengths or silently deduplicate.

Report null/negative/zero counts, exact negative values and qualifier frequencies. Any nonnegative-score analysis below is explicitly an **operational subset**, not a verified validity range. Without documentation, positive values also cannot be certified valid. Qualifiers are reported as source labels, not medical classes. Cross-tabs test whether qualifier labels accompany missing or negative scores, without displaying identities.

In [ ]:
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
audit=[]
for name,df in tables.items():
    for label,k in [('record_id',['additional_id']),('participant_date',['sdv_user_id','calendar_date']),('participant_period',key)]:
        audit.append({'table':name,'key':label,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
          'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'valid_unique_key':valid_key(df,k)})
display(pd.DataFrame(audit)); print('Exact duplicate rows after first:',int(s.duplicated().sum()))
quality=pd.DataFrame([{'field':m,'missing':int(s[m].isna().sum()),'negative':int(s[m].lt(0).sum()),
 'zero':int(s[m].eq(0).sum()),'positive':int(s[m].gt(0).sum())} for m in scores+durations])
display(quality); display(s[scores+durations].describe().T)
for score in scores:
    display(s.loc[s[score].lt(0),score].value_counts().rename_axis(score).to_frame('negative_value_records'))
qual=s.stress_qualifier.fillna('<missing>')
display(qual.value_counts(dropna=False).rename_axis('stress_qualifier').to_frame('records'))
score_status=pd.Series(np.select([s.average_stress_level.isna(),s.average_stress_level.lt(0)],
 ['missing','negative'],default='nonnegative'),index=s.index)
display(pd.crosstab(qual,score_status))
display(pd.crosstab(qual,s[durations].isna().any(axis=1)).rename_axis(columns='any_duration_missing'))
both=s[scores].notna().all(axis=1)
print('Rows with both scores present and average greater than maximum:',int((both & s.average_stress_level.gt(s.max_stress_level)).sum()))


## 3. Period quality and participant/date coverage
Duration is a reporting window, not measured wear time. Check end-minus-start consistency, nonpositive periods and overlaps. An overlap starts before the maximum preceding end for the same participant; equal endpoints are not counted under this diagnostic. Unix-seconds/UTC conversion is only a plausibility check; timezone and endpoint conventions need confirmation.

Coverage means observed dates divided by dates in each participant's first-to-last observed-date span. It cannot establish enrolment boundaries. Missing dates do not mean zero stress. Report only aggregate coverage to keep participant IDs private.

In [ ]:
s['elapsed_sec']=s.end_measure_period_sec-s.start_measure_period_sec
s['period_hours']=s.duration_measure_period_sec/3600
ordered=s.sort_values(key)
prior_end=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
utc=pd.to_datetime(s.start_measure_period_sec,unit='s',utc=True,errors='coerce')
display(pd.DataFrame([{'nonpositive_duration':int(s.duration_measure_period_sec.le(0).sum()),
 'end_not_after_start':int(s.elapsed_sec.le(0).sum()),
 'duration_matches_end_minus_start':int(s.duration_measure_period_sec.eq(s.elapsed_sec).sum()),
 'overlapping_period_rows':int(ordered.start_measure_period_sec.lt(prior_end).sum()),
 'unparseable_unix_second_starts':int(utc.isna().sum()),
 'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(s.calendar_date).sum())}]))
display(s.period_hours.value_counts().sort_index().rename_axis('hours').to_frame('records'))
coverage=s.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed=('calendar_date','nunique'))
coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
coverage['missing_dates_within_span']=coverage.span_days-coverage.observed
coverage['observed_date_percent']=100*coverage.observed/coverage.span_days
display(coverage[['observed','span_days','missing_dates_within_span','observed_date_percent']].describe().T)


## 4. Observed duration relationships, not assumed definitions
Test whether low + medium + high equals reported stress duration; then test whether stress + rest + activity equals the reporting window. These are hypotheses based on names, not established partitions. Use complete, nonnegative inputs only, count excluded rows and report residuals. Pandas sums normally skip nulls, so `min_count` is essential to prevent an incomplete sum appearing complete.

Even exact equality would show arithmetic consistency only. It would not prove categories are mutually exclusive or that every second was observed. A positive period residual is labelled **unaccounted by this candidate sum**, not missing wear time. A negative residual is not automatically a data error without definitions.

In [ ]:
checks=[]; residual_summary=[]
def check_sum(label,components,total):
    fields=list(dict.fromkeys(components+[total]))
    complete=s[fields].notna().all(axis=1)
    nonnegative=s[fields].ge(0).all(axis=1)
    use=complete & nonnegative
    candidate=s[components].sum(axis=1,min_count=len(components))
    residual=s[total]-candidate
    checks.append({'check':label,'eligible_records':int(use.sum()),'excluded_records':int((~use).sum()),
      'equal':int((use & residual.eq(0)).sum()),'total_above_sum':int((use & residual.gt(0)).sum()),
      'total_below_sum':int((use & residual.lt(0)).sum())})
    residual_summary.append(residual.loc[use].describe().rename(label))
check_sum('stress minus low+medium+high',['low_stress_duration_sec','medium_stress_duration_sec','high_stress_duration_sec'],'stress_duration_sec')
check_sum('period minus stress+rest+activity',['stress_duration_sec','rest_stress_duration_sec','activity_stress_duration_sec'],'duration_measure_period_sec')
display(pd.DataFrame(checks)); display(pd.DataFrame(residual_summary))
display(pd.DataFrame([{'duration_field':m,'complete_field_and_period':int((s[m].notna() & s.duration_measure_period_sec.notna()).sum()),
 'above_reporting_period':int(s[m].gt(s.duration_measure_period_sec).sum())} for m in durations]))


## 5. Distributions with explicit score filtering
Retain all recorded values in the earlier descriptive table. Here show nonnegative average and maximum scores separately, and complete nonnegative durations in hours. These are operational plots, not a clinical scale. Report participant medians in aggregate so unequal follow-up is visible without publishing IDs.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,m in zip(axes[:2],scores):
    s.loc[s[m].ge(0),m].plot.hist(bins=20,ax=ax)
    ax.set(title=m+' (nonnegative subset)',xlabel='Recorded score; semantics unverified',ylabel='Records')
duration_values=s[durations].where(s[durations].ge(0))/3600
axes[2].boxplot([duration_values[m].dropna() for m in durations],tick_labels=['stress','rest','low','medium','high','activity'],orientation='horizontal')
axes[2].set(title='Nonnegative recorded durations',xlabel='Hours (field-name interpretation)')
plt.tight_layout(); plt.show()
display(s.loc[s.average_stress_level.ge(0)].groupby('sdv_user_id').average_stress_level.median().describe().to_frame('participant_medians_nonnegative_average_score'))


## 6. Validated activity join and descriptive associations
Use participant/date/start/end with an outer one-to-one join, compare shared context and report unmatched records. For associations require matching group/duration, nonmissing nonnegative steps and nonmissing score. First retain **all numeric scores**, then exclude negative scores explicitly; this makes potential sentinel sensitivity visible. Also check exactly-24-hour windows and positive steps separately.

Pearson measures linear association; Spearman measures rank association. Within-participant centering subtracts each participant's subset mean before Pearson correlation. It removes mean-level differences but not repeated-measure dependence, time trends, selective missingness or unequal weighting. Participant-specific correlations need at least three records and variation. Neither strong nor weak correlation establishes cause or validates a stress algorithm.

In [ ]:
correlations=[]; participant_corr=[]
if valid_key(s,key) and valid_key(tables['activity'],key):
    j=s.merge(tables['activity'],on=key,how='outer',suffixes=('_stress','_activity'),indicator=True,validate='one_to_one')
    matched=j._merge.eq('both')
    display(j._merge.value_counts().rename_axis('join_status').to_frame('records'))
    agreement=[]
    for field in ['additional_id','sdv_group_id','duration_measure_period_sec']:
        a=j.loc[matched,field+'_stress']; b=j.loc[matched,field+'_activity']; present=a.notna() & b.notna()
        agreement.append({'field':field,'both_nonnull':int(present.sum()),'equal':int((present & a.eq(b)).sum()),
          'different':int((present & a.ne(b)).sum()),'missing_either':int((~present).sum())})
    display(pd.DataFrame(agreement))
    context=j.sdv_group_id_stress.eq(j.sdv_group_id_activity) & j.duration_measure_period_sec_stress.eq(j.duration_measure_period_sec_activity)
    numeric=matched & context & j.average_stress_level.notna() & j.steps.notna() & j.steps.ge(0)
    nonnegative=numeric & j.average_stress_level.ge(0)
    scopes={'all numeric scores, nonnegative steps':numeric,'nonnegative scores and steps':nonnegative,
      'nonnegative, known qualifier':nonnegative & j.stress_qualifier.notna() & j.stress_qualifier.ne('unknown'),
      'nonnegative, exactly 24 hours':nonnegative & j.duration_measure_period_sec_stress.eq(86400),
      'nonnegative, 24 hours, positive steps':nonnegative & j.duration_measure_period_sec_stress.eq(86400) & j.steps.gt(0)}
    print('Matched records excluded by context mismatch:',int((matched & ~context).sum()))
    print('Matched records excluded from all-numeric subset:',int((matched & ~numeric).sum()))
    print('Negative-score records removed from nonnegative subset:',int((numeric & ~nonnegative).sum()))
    def corr(d,x,y,method='pearson'):
        return d[x].corr(d[y],method=method) if len(d)>=3 and d[x].nunique()>1 and d[y].nunique()>1 else np.nan
    for label,mask in scopes.items():
        d=j.loc[mask].copy()
        for m in ['steps','average_stress_level']:
            d[m+'_centered']=d[m]-d.groupby('sdv_user_id')[m].transform('mean')
        correlations.append({'scope':label,'records':len(d),'participants':d.sdv_user_id.nunique(),
          'pooled_Pearson':corr(d,'steps','average_stress_level'),
          'pooled_Spearman':corr(d,'steps','average_stress_level','spearman'),
          'within_centered_Pearson':corr(d,'steps_centered','average_stress_level_centered')})
        for _,part in d.groupby('sdv_user_id'):
            participant_corr.append({'scope':label,'Pearson':corr(part,'steps','average_stress_level')})
    display(pd.DataFrame(correlations))
    display(pd.DataFrame(participant_corr).groupby('scope').Pearson.agg(['count','median','min','max']).rename(columns={'count':'participants_with_defined_correlation'}))
    d=j.loc[scopes['nonnegative, exactly 24 hours']].copy()
    for m in ['steps','average_stress_level']:
        d[m+'_centered']=d[m]-d.groupby('sdv_user_id')[m].transform('mean')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].scatter(d.steps,d.average_stress_level,s=15,alpha=.35)
    axes[0].set(title='24-hour windows: pooled nonnegative scores',xlabel='Recorded steps',ylabel='Recorded average stress score')
    axes[1].scatter(d.steps_centered,d.average_stress_level_centered,s=15,alpha=.35)
    axes[1].axhline(0,color='gray',lw=.7); axes[1].axvline(0,color='gray',lw=.7)
    axes[1].set(title='24-hour windows: within-participant deviations',xlabel='Steps minus participant mean',ylabel='Score minus participant mean')
    plt.tight_layout(); plt.show()
else:
    print('Activity comparison skipped: resolve nonunique or null period keys first.')


## Verified findings from this execution
- **987 records, 8 participants, 19 columns**. Tested keys are unique/non-null, and exact duplicate rows are absent. There are **217 missing cells**, including 27 missing maximum scores and 22–42 missing values per duration field.
- Average score has no nulls, but **27 records contain −1**, all with qualifier `unknown`. It is a potential sentinel, not a verified definition. There are **74 `unknown` qualifiers**, including **47 with nonnegative average scores**. Other labels are balanced, stressful, calm, calm_awake, balanced_awake, very_stressful, very_stressful_awake and stressful_awake; label meanings remain undocumented.
- No score/duration field contains recorded zero in this snapshot. Average score is the only inspected field with negative values. No complete score pair has average above maximum.
- **970 windows are exactly 24 hours**, 17 are not; all durations equal end minus start; **two overlaps** are detected. UTC start dates do not equal calendar labels under a Unix-seconds interpretation; confirm timezone/date conventions. Observed-date coverage within individual first-to-last spans ranges **89.36%–100%**, with **16–250 observed dates per participant**.
- **Low + medium + high = stress duration in all 943 complete, nonnegative records** (44 excluded). For **stress + rest + activity versus window duration**, 946 records are complete: 3 equal the window and 943 fall below it; none exceeds it. Median residual is **8,070 seconds**. That residual is not labelled missing wear time. Individual duration fields never exceed the reporting window where present.
- The exact-period one-to-one activity join matches **all 987 records**, with no unmatched keys; compared IDs, groups and durations agree.
- Retaining all numeric average scores gives pooled Pearson with steps **0.078** and within-participant-centered Pearson **0.211** (987 records). Excluding negative scores gives **0.010** pooled and **0.068** centered (960 records). In the **948 nonnegative-score exactly-24-hour records**, these are **0.010** and **0.065**, with Spearman **−0.039**. Excluding `unknown` qualifiers leaves **913 records**, with Pearson **0.027** pooled and **0.080** centered. Qualifier exclusion is sensitivity analysis, not a claim that the remaining labels certify validity.

**Takeaway:** Data-quality interpretation matters more here than the weak recorded association with steps: non-null average scores include a potential sentinel, qualifiers need definitions, and duration sums account for only part of most reporting windows. None of this establishes medical meaning or cause.


## 7. Teacher-facing method and limitations
“I checked missing and potentially sentinel scores separately, described qualifier labels without assigning medical meaning, and tested candidate duration identities using complete rows. I validated period matches before comparing average scores with activity, and separated pooled from within-participant association.”

Read each result with its eligible denominator. Arithmetic equalities are observations, not proof of duration definitions. Nonnegative scores are not automatically valid scores. Reporting duration is not wear time; missing or negative values may be selective, and participants have unequal follow-up. Device-derived stress scores and steps may share movement inputs. Repeated observations and overlap rule out treating each row as an independent person.

Ask the data owner for the stress scale, negative/sentinel codes, qualifier definitions, scoring/averaging rules, duration inclusion/overlap rules, timezone and wear-time information. No medical, causal or predictive conclusion is made. CSV export, RAG and modelling remain deferred.